# Phần 3 — Mô hình Dự báo Doanh thu (Sales Forecasting)

## Phương pháp tiếp cận

Bài toán yêu cầu dự báo **Revenue** và **COGS** hàng ngày cho giai đoạn 01/01/2023 – 01/07/2024 (548 ngày). Đây là bài toán **time-series forecasting** với các thách thức:

1. **Horizon dài** (548 ngày): Phương pháp recursive dễ tích lũy sai số → chọn **direct forecasting** (predict trực tiếp từ features)
2. **Không có auxiliary data cho test period**: Các dataset phụ (orders, payments,...) chỉ cover 2012-2022 → dùng **seasonal averages** thay vì raw values
3. **Data leakage**: Revenue/COGS từ test KHÔNG được dùng làm feature → chỉ dùng historical lags từ train

### Pipeline tổng quan

```
sales.csv + 7 auxiliary datasets
        |
        v
[1. Seasonal Feature Extraction]  -- monthly/DOW averages từ auxiliary data
        |
        v
[2. Feature Engineering]          -- 71 features: time, calendar, holiday,
        |                            cyclical, historical lags, auxiliary
        v
[3. 10-fold TimeSeriesSplit CV]   -- expanding window, no leakage
        |
        v
[4. 5-Model Ensemble]            -- LGB, XGB, HGB, CatBoost(1200), Ridge
        |
        v
[5. Ridge Stacking]              -- positive weights, alpha=1.0
        |
        v
[6. submission.csv]              -- Revenue + COGS predictions
```

### Kiểm soát Data Leakage

| Ràng buộc | Cách xử lý | Kiểm chứng |
|-----------|-----------|-----------|
| Không dùng Revenue/COGS từ test làm feature | Historical lags chỉ lookup trong `sales.csv` (train period) | Lag dates luôn < 2023-01-01 |
| Không dùng dữ liệu ngoài | Chỉ dùng 8 files CSV được cung cấp | Không import external data |
| Mã nguồn tái lập được | SEED=42 cố định, deterministic pipeline | Chạy lại cho kết quả giống nhau |
| TimeSeriesSplit | Validation set luôn SAU training set | `sklearn.TimeSeriesSplit(n_splits=10)` |
| Auxiliary data không leak | Chỉ dùng seasonal averages, không raw values cho test | Monthly/DOW means lặp lại theo chu kỳ |

**Chỉ số đánh giá**: MAE (Mean Absolute Error) — chỉ số chính trên Kaggle leaderboard.

**Kết quả**: Kaggle Public Score MAE ≈ **697,864**

## 1. Setup & Import Libraries

Các thư viện chính được sử dụng:

| Thư viện | Mục đích | Phiên bản |
|---------|---------|----------|
| **scikit-learn** | Ridge, HGB, TimeSeriesSplit, StandardScaler, metrics | ≥1.3 |
| **LightGBM** | Gradient boosting nhanh, native missing value handling | ≥4.0 |
| **XGBoost** | Gradient boosting với L1/L2/gamma regularization | ≥2.0 |
| **CatBoost** | Ordered boosting, ít overfit trên small data | ≥1.2 |
| **SHAP** | Model interpretability (TreeExplainer) | ≥0.42 |

In [ ]:
import os, warnings, random
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = "../data/"
OUTPUT_PATH = "../submission.csv"
# Kaggle:
# DATA_DIR = "/kaggle/input/competitions/datathon-2026-round-1/"
# OUTPUT_PATH = "/kaggle/working/submission.csv"
print("Setup done")

## 2. Load Data + Build Seasonal Features

### 2.1. Dữ liệu chính

| Dataset | Giai đoạn | Số dòng | Vai trò |
|---------|----------|---------|--------|
| `sales.csv` | 04/07/2012 – 31/12/2022 | 3,833 | Training data (Revenue, COGS) |
| `sample_submission.csv` | 01/01/2023 – 01/07/2024 | 548 | Test dates cần dự báo |

### 2.2. Dữ liệu phụ trợ và tương quan với Revenue

Correlation được tính bằng cách: aggregate mỗi dataset theo ngày → merge với daily Revenue → Pearson correlation.

| Dataset | Số bản ghi | Correlation với Revenue | Cách sử dụng trong pipeline |
|---------|-----------|------------------------|---------------------------|
| `orders.csv` | ~800k | **0.936** (rất cao) | Monthly avg, DOW avg, Quarterly avg, Month×DOW interaction |
| `payments.csv` | ~600k | **0.992** (cực cao) | Monthly avg, DOW avg |
| `shipments.csv` | ~500k | **0.815** (cao) | Monthly avg, DOW avg |
| `returns.csv` | ~50k | 0.441 (trung bình) | Monthly avg, Refund monthly avg |
| `web_traffic.csv` | ~3.6k | 0.321 (thấp) | Monthly avg sessions, DOW avg, Monthly visitors |
| `promotions.csv` | ~10 | — | Recurring month-day promo windows |

**Vấn đề**: Auxiliary data chỉ cover train period (2012–2022). Test period (2023–2024) KHÔNG có auxiliary data.

**Giải pháp**: Trích xuất **seasonal patterns** (trung bình theo tháng, theo ngày trong tuần). Đây là các patterns lặp lại theo chu kỳ → ngoại suy được sang test period mà không gây leakage.

### 2.3. Phương pháp trích xuất Seasonal Features

Với mỗi auxiliary dataset:
1. Aggregate theo ngày (count orders, sum payments,...)
2. Tính **monthly average** (12 giá trị) và **DOW average** (7 giá trị)
3. Map ngược về train/test theo `month` hoặc `dayofweek`
4. Đặc biệt: `orders` có thêm **quarterly avg** và **month×DOW interaction** (84 giá trị)

In [ ]:
sales = pd.read_csv(DATA_DIR + "sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sample_sub = pd.read_csv(DATA_DIR + "sample_submission.csv", parse_dates=["Date"])
orders = pd.read_csv(DATA_DIR + "orders.csv", parse_dates=["order_date"])
web = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
returns = pd.read_csv(DATA_DIR + "returns.csv", parse_dates=["return_date"])
shipments = pd.read_csv(DATA_DIR + "shipments.csv", parse_dates=["ship_date"])
payments_raw = pd.read_csv(DATA_DIR + "payments.csv")
promotions = pd.read_csv(DATA_DIR + "promotions.csv", parse_dates=["start_date", "end_date"])

print("Sales:", sales.shape, "| Test:", sample_sub.shape)
print(f"Train: {sales.Date.min().date()} to {sales.Date.max().date()}")
print(f"Test: {sample_sub.Date.min().date()} to {sample_sub.Date.max().date()}")

# --- Orders: monthly + dow + quarterly patterns ---
daily_orders = orders.groupby("order_date").agg(n_orders=("order_id", "nunique")).rename_axis("Date")
daily_orders["month"] = daily_orders.index.month
daily_orders["dayofweek"] = daily_orders.index.dayofweek
daily_orders["quarter"] = daily_orders.index.quarter
orders_month_avg = daily_orders.groupby("month")["n_orders"].mean().rename("orders_month_avg")
orders_dow_avg = daily_orders.groupby("dayofweek")["n_orders"].mean().rename("orders_dow_avg")
orders_quarter_avg = daily_orders.groupby("quarter")["n_orders"].mean().rename("orders_quarter_avg")
orders_month_dow = daily_orders.groupby(["month", "dayofweek"])["n_orders"].mean()

# --- Payments: monthly + dow ---
payments = orders[["order_id", "order_date"]].merge(payments_raw, on="order_id")
daily_pay = payments.groupby("order_date")["payment_value"].sum().rename_axis("Date").to_frame()
daily_pay["month"] = daily_pay.index.month
daily_pay["dayofweek"] = daily_pay.index.dayofweek
pay_month_avg = daily_pay.groupby("month")["payment_value"].mean()
pay_dow_avg = daily_pay.groupby("dayofweek")["payment_value"].mean()

# --- Shipments: monthly + dow ---
daily_ship = shipments.groupby("ship_date").agg(n_ship=("order_id", "nunique")).rename_axis("Date")
daily_ship["month"] = daily_ship.index.month
daily_ship["dayofweek"] = daily_ship.index.dayofweek
ship_month_avg = daily_ship.groupby("month")["n_ship"].mean()
ship_dow_avg = daily_ship.groupby("dayofweek")["n_ship"].mean()

# --- Returns: monthly ---
daily_ret = returns.groupby("return_date").agg(
    n_returns=("return_id", "nunique"), refund_total=("refund_amount", "sum")).rename_axis("Date")
daily_ret["month"] = daily_ret.index.month
ret_month_avg = daily_ret.groupby("month")["n_returns"].mean()
refund_month_avg = daily_ret.groupby("month")["refund_total"].mean()

# --- Web traffic: monthly + dow ---
web_daily = web.groupby("date").agg(
    sessions=("sessions", "sum"), unique_visitors=("unique_visitors", "sum")).rename_axis("Date")
web_daily["month"] = web_daily.index.month
web_daily["dayofweek"] = web_daily.index.dayofweek
web_month_avg = web_daily.groupby("month")["sessions"].mean()
web_dow_avg = web_daily.groupby("dayofweek")["sessions"].mean()
visitors_month_avg = web_daily.groupby("month")["unique_visitors"].mean()

# --- Revenue/COGS seasonal from sales ---
st = sales.copy()
st["month"] = st["Date"].dt.month
st["dayofweek"] = st["Date"].dt.dayofweek
rev_month_avg = st.groupby("month")["Revenue"].mean()
rev_dow_avg = st.groupby("dayofweek")["Revenue"].mean()
cogs_month_avg_s = st.groupby("month")["COGS"].mean()
margin_month_avg = (st.groupby("month")["Revenue"].mean() - st.groupby("month")["COGS"].mean()) / st.groupby("month")["Revenue"].mean()

# --- Promotions ---
promo_windows = [(r["start_date"].month, r["start_date"].day,
                  r["end_date"].month, r["end_date"].day)
                 for _, r in promotions.iterrows()]

print("Seasonal features built from all auxiliary datasets")

In [ ]:
# Tính correlation giữa auxiliary data và Revenue
# Aggregate mỗi dataset theo ngày, merge với sales, tính Pearson correlation

daily_rev = sales.set_index('Date')['Revenue']

# Orders
d_ord = orders.groupby('order_date').agg(n=('order_id','nunique')).rename_axis('Date')
corr_orders = d_ord['n'].corr(daily_rev.reindex(d_ord.index))

# Payments
d_pay = orders[['order_id','order_date']].merge(payments_raw, on='order_id')
d_pay = d_pay.groupby('order_date')['payment_value'].sum().rename_axis('Date')
corr_payments = d_pay.corr(daily_rev.reindex(d_pay.index))

# Shipments
d_ship = shipments.groupby('ship_date').agg(n=('order_id','nunique')).rename_axis('Date')
corr_shipments = d_ship['n'].corr(daily_rev.reindex(d_ship.index))

# Returns
d_ret = returns.groupby('return_date').agg(n=('return_id','nunique')).rename_axis('Date')
corr_returns = d_ret['n'].corr(daily_rev.reindex(d_ret.index))

# Web traffic
d_web = web.groupby('date').agg(s=('sessions','sum')).rename_axis('Date')
corr_web = d_web['s'].corr(daily_rev.reindex(d_web.index))

print('=== Correlation với Revenue (Pearson) ===')
print(f'  orders.csv      : {corr_orders:.3f}  ({len(orders):,} records)')
print(f'  payments.csv    : {corr_payments:.3f}  ({len(payments_raw):,} records)')
print(f'  shipments.csv   : {corr_shipments:.3f}  ({len(shipments):,} records)')
print(f'  returns.csv     : {corr_returns:.3f}  ({len(returns):,} records)')
print(f'  web_traffic.csv : {corr_web:.3f}  ({len(web):,} records)')
print(f'  promotions.csv  : —       ({len(promotions):,} records)')

## 3. Feature Engineering — 71 Features

Features được thiết kế theo 6 nhóm, mỗi nhóm capture một khía cạnh khác nhau của Revenue pattern:

### 3.1. Time Features (12 features)

Các thành phần thời gian cơ bản: `year`, `month`, `day`, `dayofweek`, `dayofyear`, `weekofyear`, `quarter`, `is_weekend`, `is_month_start`, `is_month_end`, `time_idx` (linear trend), `day_of_month_pct`.

### 3.2. Calendar & Holiday Features (19 features)

Nhóm features mới giúp capture các business patterns:

| Nhóm | Features | Lý do |
|------|----------|-------|
| Ngày cụ thể | is_saturday, is_sunday, is_monday, is_friday | Revenue khác biệt rõ rệt giữa các ngày |
| Vị trí trong tháng | week_of_month, is_first_week, is_last_week, is_mid_month, days_to_month_end | Pattern đầu/cuối tháng (lương, thanh toán) |
| Quý | is_q1–q4, is_quarter_start/end, month_in_quarter | Seasonal pattern theo quý |
| Năm | is_year_start, is_year_end | Revenue đặc biệt đầu/cuối năm |
| Ngày lễ | is_holiday, is_near_holiday (±3 ngày) | Tết, 30/4, 1/5, 2/9, Giáng sinh |

**Impact**: Thêm nhóm này giảm MAE từ 707k → 700k (giảm 7k).

### 3.3. Cyclical Features (10 features)

Sin/cos encoding cho `month` (period=12), `dayofweek` (period=7), `dayofyear` (3 harmonics, period=365.25). Giúp model hiểu tháng 12 và tháng 1 gần nhau trong chu kỳ.

### 3.4. Historical Lag Features (10 features)

| Feature | Công thức | Xử lý missing |
|---------|----------|--------------|
| rev_1y/2y/3y | Revenue cùng ngày ±1 ngày, 1/2/3 năm trước | Cascade: 1y→2y→3y→median |
| cogs_1y/2y/3y | COGS tương tự | Cascade fill |
| rev_yoy_growth | (rev_1y − rev_2y) / (rev_2y + 1) | — |
| margin_1y | (rev_1y − cogs_1y) / (rev_1y + 1) | — |
| rev_avg_hist | mean(rev_1y, rev_2y, rev_3y) | Fill 0 trước mean |
| rev_1y_vs_month | rev_1y / (monthly_avg + 1) | — |

**Leakage check**: Cho test date 2023-03-15, lag 365 = 2022-03-15 (trong train ✓). Cho 2024-03-15, lag 365 = 2023-03-15 (KHÔNG có) → dùng lag 730 = 2022-03-15 (trong train ✓).

### 3.5. Auxiliary Seasonal Features (19 features)

Monthly/DOW averages từ 6 auxiliary datasets + Revenue/COGS seasonal patterns từ train data. Chi tiết: orders (4), payments (2), shipments (2), returns (2), web (3), revenue/cogs seasonal (6), promotions (1).

### 3.6. Xử lý Missing Values

- Historical lags: **Cascade fill** (1y → 2y → 3y → median)
- NaN còn lại: fill với **median từ train only** (không dùng test data)
- Giá trị Inf: replace bằng 0

In [ ]:
def is_in_promo(date):
    md_val = (date.month, date.day)
    for sm, sd, em, ed in promo_windows:
        start, end = (sm, sd), (em, ed)
        if start <= end:
            if start <= md_val <= end:
                return 1
        else:
            if md_val >= start or md_val <= end:
                return 1
    return 0

# Vietnamese holidays (month, day)
VN_HOLIDAYS = [
    (1,1),   # Tet Duong lich
    (4,30),  # Giai phong Mien Nam
    (5,1),   # Quoc te Lao dong
    (9,2),   # Quoc khanh
    (12,25), # Giang sinh
]
# Tet Am lich (approximate dates - varies each year)
TET_DATES = [
    (2013,2,9),(2013,2,10),(2013,2,11),(2013,2,12),(2013,2,13),
    (2014,1,30),(2014,1,31),(2014,2,1),(2014,2,2),(2014,2,3),
    (2015,2,18),(2015,2,19),(2015,2,20),(2015,2,21),(2015,2,22),
    (2016,2,7),(2016,2,8),(2016,2,9),(2016,2,10),(2016,2,11),
    (2017,1,27),(2017,1,28),(2017,1,29),(2017,1,30),(2017,1,31),
    (2018,2,15),(2018,2,16),(2018,2,17),(2018,2,18),(2018,2,19),
    (2019,2,4),(2019,2,5),(2019,2,6),(2019,2,7),(2019,2,8),
    (2020,1,24),(2020,1,25),(2020,1,26),(2020,1,27),(2020,1,28),
    (2021,2,11),(2021,2,12),(2021,2,13),(2021,2,14),(2021,2,15),
    (2022,1,31),(2022,2,1),(2022,2,2),(2022,2,3),(2022,2,4),
    (2023,1,21),(2023,1,22),(2023,1,23),(2023,1,24),(2023,1,25),
    (2024,2,9),(2024,2,10),(2024,2,11),(2024,2,12),(2024,2,13),
]
tet_set = set((y,m,d) for y,m,d in TET_DATES)

def is_holiday(date):
    if (date.month, date.day) in VN_HOLIDAYS:
        return 1
    if (date.year, date.month, date.day) in tet_set:
        return 1
    return 0

def is_near_holiday(date, window=3):
    for delta in range(-window, window+1):
        d2 = date + pd.Timedelta(days=delta)
        if is_holiday(d2):
            return 1
    return 0

# Revenue by month x dow interaction
rev_month_dow = st.groupby(["month", "dayofweek"])["Revenue"].mean()
rev_month_std = st.groupby("month")["Revenue"].std()
rev_month_med = st.groupby("month")["Revenue"].median()

def create_features(df, sales_full, min_date):
    df = df.copy()
    # === Time features ===
    df["year"] = df["Date"].dt.year
    df["month"] = df["Date"].dt.month
    df["day"] = df["Date"].dt.day
    df["dayofweek"] = df["Date"].dt.dayofweek
    df["dayofyear"] = df["Date"].dt.dayofyear
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["Date"].dt.quarter
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = df["Date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["Date"].dt.is_month_end.astype(int)
    df["time_idx"] = (df["Date"] - min_date).dt.days
    df["day_of_month_pct"] = df["day"] / df["Date"].dt.days_in_month

    # === NEW: Calendar features ===
    df["is_saturday"] = (df["dayofweek"] == 5).astype(int)
    df["is_sunday"] = (df["dayofweek"] == 6).astype(int)
    df["is_monday"] = (df["dayofweek"] == 0).astype(int)
    df["is_friday"] = (df["dayofweek"] == 4).astype(int)
    df["week_of_month"] = (df["day"] - 1) // 7 + 1
    df["is_first_week"] = (df["day"] <= 7).astype(int)
    df["is_last_week"] = (df["day"] >= df["Date"].dt.days_in_month - 6).astype(int)
    df["is_mid_month"] = ((df["day"] >= 13) & (df["day"] <= 17)).astype(int)
    df["days_to_month_end"] = df["Date"].dt.days_in_month - df["day"]
    # Quarter features
    df["is_q1"] = (df["quarter"] == 1).astype(int)
    df["is_q2"] = (df["quarter"] == 2).astype(int)
    df["is_q3"] = (df["quarter"] == 3).astype(int)
    df["is_q4"] = (df["quarter"] == 4).astype(int)
    df["is_quarter_start"] = df["Date"].dt.is_quarter_start.astype(int)
    df["is_quarter_end"] = df["Date"].dt.is_quarter_end.astype(int)
    df["month_in_quarter"] = (df["month"] - 1) % 3 + 1
    # Year features
    df["is_year_start"] = df["Date"].dt.is_year_start.astype(int)
    df["is_year_end"] = df["Date"].dt.is_year_end.astype(int)
    # Holidays
    df["is_holiday"] = df["Date"].apply(is_holiday)
    df["is_near_holiday"] = df["Date"].apply(is_near_holiday)

    # === Cyclical features (3 harmonics) ===
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    for k in [1, 2, 3]:
        df[f"doy_sin{k}"] = np.sin(2 * k * np.pi * df["dayofyear"] / 365.25)
        df[f"doy_cos{k}"] = np.cos(2 * k * np.pi * df["dayofyear"] / 365.25)

    # === Historical Revenue/COGS ===
    rev_s = sales_full.set_index("Date")["Revenue"]
    cogs_s = sales_full.set_index("Date")["COGS"]
    for lag in [364, 365, 366, 728, 729, 730, 731, 1095, 1096]:
        lag_dates = df["Date"] - pd.Timedelta(days=lag)
        df[f"rev_lag_{lag}"] = lag_dates.map(rev_s).values
        df[f"cogs_lag_{lag}"] = lag_dates.map(cogs_s).values

    df["rev_1y"] = df["rev_lag_365"].fillna(df["rev_lag_366"]).fillna(df["rev_lag_364"])
    df["rev_2y"] = df["rev_lag_730"].fillna(df["rev_lag_731"]).fillna(df["rev_lag_729"]).fillna(df["rev_lag_728"])
    df["rev_3y"] = df["rev_lag_1095"].fillna(df["rev_lag_1096"])
    df["cogs_1y"] = df["cogs_lag_365"].fillna(df["cogs_lag_366"]).fillna(df["cogs_lag_364"])
    df["cogs_2y"] = df["cogs_lag_730"].fillna(df["cogs_lag_731"]).fillna(df["cogs_lag_729"]).fillna(df["cogs_lag_728"])
    df["cogs_3y"] = df["cogs_lag_1095"].fillna(df["cogs_lag_1096"])

    df["rev_1y"] = df["rev_1y"].fillna(df["rev_2y"]).fillna(df["rev_3y"])
    df["rev_2y"] = df["rev_2y"].fillna(df["rev_3y"])
    df["cogs_1y"] = df["cogs_1y"].fillna(df["cogs_2y"]).fillna(df["cogs_3y"])
    df["cogs_2y"] = df["cogs_2y"].fillna(df["cogs_3y"])

    df["rev_yoy_growth"] = (df["rev_1y"] - df["rev_2y"]) / (df["rev_2y"] + 1)
    df["margin_1y"] = (df["rev_1y"] - df["cogs_1y"]) / (df["rev_1y"] + 1)
    # NEW: avg of historical years
    df["rev_avg_hist"] = (df["rev_1y"].fillna(0) + df["rev_2y"].fillna(0) + df["rev_3y"].fillna(0)) / 3
    df["rev_1y_vs_month"] = df["rev_1y"] / (df["month"].map(rev_month_avg) + 1)

    # === Auxiliary seasonal features ===
    df["orders_month_avg"] = df["month"].map(orders_month_avg)
    df["orders_dow_avg"] = df["dayofweek"].map(orders_dow_avg)
    df["orders_quarter_avg"] = df["quarter"].map(orders_quarter_avg)
    df["orders_month_dow"] = df.apply(
        lambda r: orders_month_dow.get((r["month"], r["dayofweek"]), np.nan), axis=1)
    df["pay_month_avg"] = df["month"].map(pay_month_avg)
    df["pay_dow_avg"] = df["dayofweek"].map(pay_dow_avg)
    df["ship_month_avg"] = df["month"].map(ship_month_avg)
    df["ship_dow_avg"] = df["dayofweek"].map(ship_dow_avg)
    df["ret_month_avg"] = df["month"].map(ret_month_avg)
    df["refund_month_avg"] = df["month"].map(refund_month_avg)
    df["web_month_avg"] = df["month"].map(web_month_avg)
    df["web_dow_avg"] = df["dayofweek"].map(web_dow_avg)
    df["visitors_month_avg"] = df["month"].map(visitors_month_avg)

    # Revenue/COGS seasonal
    df["rev_month_avg"] = df["month"].map(rev_month_avg)
    df["rev_dow_avg"] = df["dayofweek"].map(rev_dow_avg)
    df["cogs_month_avg"] = df["month"].map(cogs_month_avg_s)
    df["margin_month_avg"] = df["month"].map(margin_month_avg)
    # NEW: month x dow revenue + stats
    df["rev_month_dow"] = df.apply(lambda r: rev_month_dow.get((r["month"], r["dayofweek"]), np.nan), axis=1)
    df["rev_month_std"] = df["month"].map(rev_month_std)
    df["rev_month_med"] = df["month"].map(rev_month_med)


    # Promotions
    df["is_promo"] = df["Date"].apply(is_in_promo)
    return df

min_date = sales["Date"].min()
train_df = create_features(sales, sales, min_date)
test_df = create_features(sample_sub[["Date"]], sales, min_date)

feature_cols = [
    # Time (12)
    "year", "month", "day", "dayofweek", "dayofyear", "weekofyear", "quarter",
    "is_weekend", "is_month_start", "is_month_end", "time_idx", "day_of_month_pct",
    # NEW Calendar (17)
    "is_saturday", "is_sunday", "is_monday", "is_friday",
    "week_of_month", "is_first_week", "is_last_week", "is_mid_month", "days_to_month_end",
    "is_q1", "is_q2", "is_q3", "is_q4",
    "is_quarter_start", "is_quarter_end", "month_in_quarter",
    "is_year_start", "is_year_end",
    # NEW Holidays (2)
    "is_holiday", "is_near_holiday",
    # Cyclical (10)
    "month_sin", "month_cos", "dow_sin", "dow_cos",
    "doy_sin1", "doy_cos1", "doy_sin2", "doy_cos2", "doy_sin3", "doy_cos3",
    # Historical (10)
    "rev_1y", "rev_2y", "rev_3y", "cogs_1y", "cogs_2y", "cogs_3y",
    "rev_yoy_growth", "margin_1y", "rev_avg_hist", "rev_1y_vs_month",
    # Auxiliary (22)
    "orders_month_avg", "orders_dow_avg", "orders_quarter_avg", "orders_month_dow",
    "pay_month_avg", "pay_dow_avg",
    "ship_month_avg", "ship_dow_avg",
    "ret_month_avg", "refund_month_avg",
    "web_month_avg", "web_dow_avg", "visitors_month_avg",
    "rev_month_avg", "rev_dow_avg", "cogs_month_avg", "margin_month_avg",
    "rev_month_dow", "rev_month_std", "rev_month_med",
    "is_promo",

]

# Fill NaN with median from train
for col in feature_cols:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)
train_df = train_df.replace([np.inf, -np.inf], 0)
test_df = test_df.replace([np.inf, -np.inf], 0)

X_full = train_df[feature_cols]
y_rev = train_df["Revenue"]
y_cogs = train_df["COGS"]
X_test = test_df[feature_cols]

print(f"Features: {len(feature_cols)} | Train: {X_full.shape} | Test: {X_test.shape}")
print(f"NaN train: {X_full.isna().any().any()} | NaN test: {X_test.isna().any().any()}")

## 4. Cross-Validation + Model Training

### 4.1. Chiến lược Cross-Validation: TimeSeriesSplit

```
Fold 1:  [===TRAIN===][VAL]                    ← ít data nhất
Fold 2:  [=====TRAIN=====][VAL]
Fold 3:  [========TRAIN========][VAL]
...
Fold 10: [==================TRAIN==================][VAL] ← nhiều data nhất
```

- **10 folds** với expanding window
- Validation set **luôn nằm SAU** training set → không data leakage
- OOF predictions dùng cho stacking (Level 1)
- Test predictions = **trung bình 10 folds** → giảm variance

### 4.2. Ensemble: 5 Base Models

Sử dụng 5 models đa dạng (tree-based + linear) để giảm variance:

| Model | Iterations | Key Hyperparameters | Đặc điểm |
|-------|-----------|-------------------|---------|
| **LightGBM** | 800 | depth=8, leaves=95, lr=0.02 | Nhanh, xử lý missing |
| **XGBoost** | 800 | depth=7, gamma=0.1, lr=0.02 | L1+L2+gamma regularization |
| **HGB** | 800 | depth=8, L2=1.5, lr=0.02 | Sklearn native, ổn định |
| **CatBoost** | **1200** | depth=8, L2=5.0, lr=0.02 | Ordered boosting, **iterations cao nhất** |
| **Ridge** | — | alpha=10 + StandardScaler | Linear, bắt trend dài hạn |

**Tại sao CatBoost 1200 iters?** Phân tích stacking weights cho thấy CatBoost chiếm ~92% weight → tăng iterations giúp model này học kỹ hơn. Thực nghiệm: 800→1200 iters giảm MAE từ 700k→697.8k.

### 4.3. Target Transform

- **`y = log1p(Revenue)`**: Giảm skewness, giúp model xử lý outliers tốt hơn
- **Inverse**: `Revenue = expm1(y_pred)`, clip ≥ 0
- Lý do: Revenue distribution có right skew mạnh (nhiều ngày doanh thu thấp, ít ngày doanh thu rất cao)

In [ ]:
def make_models(seed=42):
    return {
        "lgb": LGBMRegressor(
            n_estimators=800, learning_rate=0.02, max_depth=8,
            num_leaves=95, min_child_samples=15, subsample=0.75,
            colsample_bytree=0.75, reg_alpha=0.05, reg_lambda=1.5,
            random_state=seed, verbose=-1, n_jobs=-1),
        "xgb": XGBRegressor(
            n_estimators=800, learning_rate=0.02, max_depth=7,
            min_child_weight=5, subsample=0.75, colsample_bytree=0.75,
            reg_alpha=0.05, reg_lambda=1.5, gamma=0.1,
            random_state=seed, verbosity=0, n_jobs=-1),
        "hgb": HistGradientBoostingRegressor(
            max_iter=800, learning_rate=0.02, max_depth=8,
            max_leaf_nodes=95, min_samples_leaf=15,
            l2_regularization=1.5, random_state=seed),
        "cat": CatBoostRegressor(
            iterations=1200, learning_rate=0.02, depth=8,
            l2_leaf_reg=5.0, random_seed=seed, verbose=0),
        "ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0))]),
    }

N_FOLDS = 10
tscv = TimeSeriesSplit(n_splits=N_FOLDS)
model_names = list(make_models().keys())

oof_rev = {n: np.zeros(len(X_full)) for n in model_names}
oof_cogs = {n: np.zeros(len(X_full)) for n in model_names}
oof_mask = np.zeros(len(X_full), dtype=bool)
test_rev_preds = {n: np.zeros(len(X_test)) for n in model_names}
test_cogs_preds = {n: np.zeros(len(X_test)) for n in model_names}

y_rev_log = np.log1p(y_rev)
y_cogs_log = np.log1p(y_cogs)

for fold_i, (tr_idx, val_idx) in enumerate(tscv.split(X_full)):
    print(f"\n=== Fold {fold_i+1}/{N_FOLDS} (train: {len(tr_idx)}, val: {len(val_idx)}) ===")
    X_tr, X_val = X_full.iloc[tr_idx], X_full.iloc[val_idx]
    yr_tr = y_rev_log.iloc[tr_idx]
    yc_tr = y_cogs_log.iloc[tr_idx]
    oof_mask[val_idx] = True

    for name in model_names:
        m_rev = make_models(SEED + fold_i)[name]
        m_rev.fit(X_tr, yr_tr)
        p_val = np.maximum(np.expm1(m_rev.predict(X_val)), 0)
        p_test = np.maximum(np.expm1(m_rev.predict(X_test)), 0)
        oof_rev[name][val_idx] = p_val
        test_rev_preds[name] += p_test / N_FOLDS

        m_cogs = make_models(SEED + fold_i + 100)[name]
        m_cogs.fit(X_tr, yc_tr)
        pc_val = np.maximum(np.expm1(m_cogs.predict(X_val)), 0)
        pc_test = np.maximum(np.expm1(m_cogs.predict(X_test)), 0)
        oof_cogs[name][val_idx] = pc_val
        test_cogs_preds[name] += pc_test / N_FOLDS

        mae_r = mean_absolute_error(y_rev.iloc[val_idx], p_val)
        print(f"  {name:5s} | Rev MAE: {mae_r:>12,.0f}")

print("\n=== OOF Results ===")
mask = oof_mask
for name in model_names:
    mae_r = mean_absolute_error(y_rev[mask], oof_rev[name][mask])
    r2_r = r2_score(y_rev[mask], oof_rev[name][mask])
    print(f"  {name:5s} Rev MAE: {mae_r:>12,.0f} | R2: {r2_r:.4f}")

## 5. Stacking — Ridge Meta-Learner

### 5.1. Kiến trúc 2 lớp

```
Level 0:  LGB_oof  XGB_oof  HGB_oof  Cat_oof  Ridge_oof
              \       |        |        |        /
               \      |        |        |       /
          Ridge(alpha=1.0, positive=True)       ← Level 1
                         |
                  Final Prediction
```

### 5.2. Thiết kế Meta-Learner

| Quyết định | Lý do |
|-----------|-------|
| **Ridge** (không phải tree-based) | Đơn giản hơn base models → tránh overfitting lớp 2 |
| **positive=True** | Trọng số ≥ 0 → interpretable, tránh negative weights gây bất ổn |
| **alpha=1.0** | Regularization vừa phải |

### 5.3. Phân tích Stacking Weights

| Model | Weight | Nhận xét |
|-------|--------|---------|
| CatBoost | ~0.92 | Model mạnh nhất, chiếm gần toàn bộ |
| HGB | ~0.11 | Bổ trợ nhỏ |
| Ridge | ~0.02 | Điều chỉnh trend |
| LGB | ~0.00 | Không trực tiếp đóng góp |
| XGB | ~0.00 | Không trực tiếp đóng góp |

**Lưu ý**: Dù LGB/XGB có weight ~0, bỏ chúng ra khỏi ensemble khiến MAE tăng (703k vs 697.8k). Điều này cho thấy diversity vẫn có giá trị gián tiếp trong stacking.

### 5.4. COGS Estimation

COGS được predict tương tự Revenue (cùng pipeline, cùng models). Final COGS = 70% model prediction + 30% ratio-based (COGS = Revenue × historical COGS/Revenue ratio).

In [ ]:
mask = oof_mask
stack_tr_rev = pd.DataFrame({n: oof_rev[n][mask] for n in model_names})
stack_tr_cogs = pd.DataFrame({n: oof_cogs[n][mask] for n in model_names})
stack_te_rev = pd.DataFrame({n: test_rev_preds[n] for n in model_names})
stack_te_cogs = pd.DataFrame({n: test_cogs_preds[n] for n in model_names})

meta_rev = Ridge(alpha=1.0, positive=True)
meta_rev.fit(stack_tr_rev, y_rev[mask].values)
meta_cogs = Ridge(alpha=1.0, positive=True)
meta_cogs.fit(stack_tr_cogs, y_cogs[mask].values)

print("Revenue stacking weights:")
for n, w in zip(model_names, meta_rev.coef_):
    print(f"  {n:5s}: {w:.4f}")

stacked_oof = meta_rev.predict(stack_tr_rev)
mae = mean_absolute_error(y_rev[mask], stacked_oof)
rmse = np.sqrt(mean_squared_error(y_rev[mask], stacked_oof))
r2 = r2_score(y_rev[mask], stacked_oof)
print(f"\nStacked OOF Revenue: MAE={mae:,.0f} | RMSE={rmse:,.0f} | R2={r2:.4f}")

## 6. Final Predictions + Submission

- Output: `submission.csv` với 548 dòng (01/01/2023 – 01/07/2024)
- Format: `Date, Revenue, COGS`
- Revenue ≥ 0 (clipped), COGS blend model + ratio

In [ ]:
final_revenue = np.maximum(meta_rev.predict(stack_te_rev), 0)
final_cogs_raw = np.maximum(meta_cogs.predict(stack_te_cogs), 0)

# Blend COGS with ratio-based fallback
cogs_ratio = sales["COGS"].sum() / sales["Revenue"].sum()
ratio_cogs = final_revenue * cogs_ratio
final_cogs = 0.7 * final_cogs_raw + 0.3 * ratio_cogs

print(f"Revenue range: {final_revenue.min():.0f} ~ {final_revenue.max():.0f}")
print(f"COGS range: {final_cogs.min():.0f} ~ {final_cogs.max():.0f}")

submission = pd.DataFrame({
    "Date": sample_sub["Date"].dt.strftime("%Y-%m-%d"),
    "Revenue": final_revenue.round(2),
    "COGS": final_cogs.round(2),
})

display(submission.head(10))
display(submission.tail(10))
print(f"Shape: {submission.shape} | Missing: {submission.isna().sum().sum()}")

submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

## 7. Giải thích Mô hình (Model Interpretability)

### 7.1. Feature Importance — LightGBM Gain

Gain-based importance đo **tổng gain (giảm loss)** khi feature được dùng để split. Feature có gain cao = đóng góp nhiều vào prediction accuracy.

**Expected top features**: `time_idx` (trend), `rev_1y`/`cogs_1y` (historical), `ord_md` (order patterns), cyclical features.

In [ ]:
# Train LGB on full data for importance analysis
import matplotlib.pyplot as plt

final_lgb = LGBMRegressor(n_estimators=800, learning_rate=0.02, max_depth=8,
    num_leaves=95, min_child_samples=15, subsample=0.75, colsample_bytree=0.75,
    reg_alpha=0.05, reg_lambda=1.5, random_state=SEED, verbose=-1, n_jobs=-1)
final_lgb.fit(X_full, np.log1p(y_rev))

imp = pd.DataFrame({"feature": feature_cols, "importance": final_lgb.feature_importances_})
imp = imp.sort_values("importance", ascending=False)

plt.figure(figsize=(10, 13))
plt.barh(imp["feature"].values[::-1], imp["importance"].values[::-1])
plt.title("LightGBM Feature Importance (Gain)", fontsize=14)
plt.xlabel("Importance (Gain)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("\nTop 15 Features:")
display(imp.head(15))
print("\nBottom 5 Features (it quan trong nhat):")
display(imp.tail(5))

### 7.2. SHAP Values — Per-sample Explanation

SHAP (SHapley Additive exPlanations) phân tích đóng góp của **từng feature cho từng sample**:

- **Trục X**: SHAP value (dương = tăng Revenue, âm = giảm Revenue)
- **Màu đỏ**: Feature value cao, **Màu xanh**: Feature value thấp
- **Vị trí trên/dưới**: Feature ở trên có mean |SHAP| lớn hơn = quan trọng hơn

SHAP giúp trả lời: "Tại sao model dự báo Revenue cao/thấp cho ngày cụ thể?"

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(final_lgb)
    sample_X = X_full.sample(500, random_state=42)
    shap_vals = explainer.shap_values(sample_X)

    plt.figure(figsize=(10, 13))
    shap.summary_plot(shap_vals, sample_X, feature_names=feature_cols, show=False)
    plt.title("SHAP Summary Plot - Feature Impact on Revenue Prediction")
    plt.tight_layout()
    plt.show()
    print("SHAP analysis completed successfully")
except ImportError:
    print("Thu vien shap chua duoc cai dat.")
    print("Cai dat bang: pip install shap")
    print("Feature importance (gain-based) da duoc hien thi o tren.")

## 8. Kết quả Thực nghiệm và Tổng kết

### 8.1. Tuân thủ ràng buộc

| Ràng buộc | Trạng thái | Minh chứng |
|-----------|-----------|-----------|
| Không dùng Revenue/COGS từ test làm feature | ✅ | Lag features chỉ lookup train dates (< 2023-01-01) |
| Không dùng dữ liệu ngoài | ✅ | Chỉ dùng 8 CSV files trong data/ |
| Mã nguồn tái lập được | ✅ | SEED=42, deterministic, notebook chạy lại cho cùng kết quả |

### 8.2. Kết quả trên Kaggle Leaderboard

| Metric | Giá trị |
|--------|--------|
| **MAE** | **697,864** |
| OOF MAE (10-fold CV) | Xem output Section 4 & 5 |

### 8.3. Quá trình tối ưu (Ablation Study)

| Thay đổi | Kaggle MAE | Delta |
|----------|-----------|-------|
| Baseline (48 features, 5 models × 800 iters) | 707,000 | — |
| +23 Calendar/Holiday features (→ 71 features) | 700,000 | **−7k** |
| +CatBoost 1200 iters (thay vì 800) | **697,864** | **−2.1k** |

### 8.4. Thực nghiệm không thành công (Negative Results)

| Thay đổi | Kết quả | Nguyên nhân |
|----------|--------|------------|
| Thêm 18 features nữa (→ 89 features) | 718k (+18k) | Noise từ features yếu |
| Bỏ LGB/XGB (chỉ Cat+HGB+Ridge) | 703k (+5k) | Mất diversity |
| Tăng tất cả iterations (1200/1600) | 700k (+2k) | Overfitting nhẹ |
| Regularization mạnh (depth=6, leaves=50) | ~700k | Underfitting |
| Prophet + Q-Specialist blend | ~1,000k | Blend sai hướng |

### 8.5. Các điểm mạnh của Pipeline

1. **Direct forecasting**: Predict trực tiếp 548 ngày từ features, không recursive → tránh tích lũy sai số
2. **71 features từ 7 nguồn**: Kết hợp time, calendar, holiday, cyclical, historical lags, auxiliary seasonal
3. **5-model ensemble + Ridge stacking**: Tăng robustness, giảm variance, tự động tìm weights tối ưu
4. **10-fold TimeSeriesSplit**: Đánh giá nghiêm ngặt, không data leakage
5. **SHAP + Feature Importance**: Giải thích rõ model đang học gì, features nào quan trọng

### 8.6. Key Insights

- **Historical Revenue cùng kỳ** (`rev_1y`, `rev_2y`) là features quan trọng nhất → Revenue có tính seasonal mạnh
- **CatBoost chiếm 92% stacking weight** → ordered boosting phù hợp nhất với time-series data nhỏ (3,833 samples)
- **Calendar features** (ngày lễ, quý, tuần) cải thiện MAE 7k → business cycles ảnh hưởng rõ rệt
- **Log1p transform** giúp model xử lý outliers Revenue
- **Diversity quan trọng**: Dù LGB/XGB weight ~0, bỏ chúng làm MAE tăng 5k